# 01 — Coin Flip Betting & the Kelly Criterion

We simulate repeated bets on a biased coin, where each round a strategy decides what fraction of current capital to stake. We compare fixed-fraction, Kelly, half-Kelly, martingale, and all-in strategies, then look at the expected log-growth curve and empirical ruin probabilities.

Setup: coin wins with probability `p`, a win pays `b` times the stake (net odds), a loss forfeits the stake.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quant_sims.betting import (
    CoinFlipGame,
    kelly_fraction,
    growth_rate_curve,
    fixed_fraction,
    kelly_strategy,
    martingale,
    all_in,
    ruin_probability_montecarlo,
)
from quant_sims.utils.plotting import (
    plot_equity_curves,
    plot_final_capital_distribution,
    plot_strategy_comparison,
    plot_growth_rate_curve,
)

%matplotlib inline

## 1. Define the edge

We use a coin with a modest positive edge: p = 0.55, even-money payout (b = 1.0).

In [ ]:
p, b = 0.55, 1.0
f_star = kelly_fraction(p, b)
print(f"Kelly-optimal fraction f* = {f_star:.4f}")

## 2. Expected log-growth curve

Sweep bet fraction f and plot expected per-bet log growth. The Kelly fraction should sit exactly at the maximum.

In [ ]:
f_values = np.linspace(-0.2, 0.99, 400)
growth_rates = growth_rate_curve(p, b, f_values)
plot_growth_rate_curve(f_values, growth_rates, f_star)
plt.show()

## 3. Monte Carlo: single strategy equity curves

In [ ]:
game = CoinFlipGame(p_win=p, payout_ratio=b, seed=42)
strategy = kelly_strategy(p, b)

paths = game.monte_carlo(strategy, n_flips=200, n_simulations=500, initial_capital=100.0)
plot_equity_curves(paths, title="Full Kelly — sample equity curves")
plt.show()

plot_final_capital_distribution(paths, title="Full Kelly — final capital distribution")
plt.show()

## 4. Comparing strategies

Same coin, same random seed pattern, different staking strategies. Note the log y-axis: Kelly is designed to maximize long-run *median* growth, not necessarily *mean* growth, and all-in strategies go to ruin almost surely given enough flips.

In [ ]:
n_flips, n_sims, initial_capital = 200, 500, 100.0

strategies = {
    "Fixed 5%": fixed_fraction(0.05),
    "Full Kelly": kelly_strategy(p, b, multiplier=1.0),
    "Half Kelly": kelly_strategy(p, b, multiplier=0.5),
    "Martingale (base 2%)": martingale(0.02),
    "All-in": all_in(),
}

paths_by_strategy = {}
for name, strat in strategies.items():
    g = CoinFlipGame(p_win=p, payout_ratio=b, seed=42)
    paths_by_strategy[name] = g.monte_carlo(strat, n_flips=n_flips, n_simulations=n_sims, initial_capital=initial_capital)

plot_strategy_comparison(paths_by_strategy)
plt.show()

## 5. Ruin probability by strategy

Probability that capital ever drops to 5% of its starting value within the simulation window.

In [ ]:
for name, strat in strategies.items():
    g = CoinFlipGame(p_win=p, payout_ratio=b, seed=42)
    ruin_prob = ruin_probability_montecarlo(g, strat, n_flips=n_flips, n_simulations=300, initial_capital=initial_capital)
    print(f"{name:25s} ruin probability = {ruin_prob:.2%}")

## Takeaways

- The Kelly fraction sits exactly at the peak of the expected-log-growth curve, by construction.
- Full Kelly grows fastest in the long run (median terms) but with substantial variance along the way.
- Half-Kelly trades some growth for meaningfully lower drawdown risk — a common real-world compromise.
- Martingale and all-in strategies look tempting on lucky paths but carry high ruin probability; they do not have positive risk-adjusted long-run growth despite the coin having a genuine edge.
- Even with a *positive* edge, bet sizing dominates outcomes as much as the edge itself.